# Week 10 Project 3
Student: Daniel Foulen

Class: IS 362

Date: 4/15/2026

Query a SQL database of music purchase records into a pandas DataFrame. This reminds me a lot of LastFM. 

I'm glad that my top track for this week isn't 2016 meme music anymore.

## Notes

The original Chinook database link from the assignment (chinookdatabase.codeplex.com) 
is no longer active. This notebook fetches the database from the maintained GitHub 
mirror at github.com/lerocha/chinook-database.

## Step 1: Download the Database

The [Chinook database](https://github.com/lerocha/chinook-database) is a sample SQLite database modelling a digital music store. We fetch the pre-built `.sqlite` file using `urllib.request` from the standard library. An existence check prevents a redundant re-download on subsequent runs.

In [1]:
import os
import urllib.request

DB_FILE = "Chinook_Sqlite.sqlite"
DB_URL  = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"

if os.path.exists(DB_FILE):
    print(f"{DB_FILE} already exists, skipping download.")
else:
    print(f"Downloading {DB_FILE}...")
    urllib.request.urlretrieve(DB_URL, DB_FILE)
    print(f"Done. ({os.path.getsize(DB_FILE):,} bytes)")

Chinook_Sqlite.sqlite already exists, skipping download.


## Step 2: Imports

`sqlite3` is part of the Python standard library and gives us a lightweight connection to any SQLite file. `pandas` wraps that connection in `read_sql()`, which executes a query and returns the result directly as a DataFrame.

In [2]:
import sqlite3
import pandas as pd

## Step 3: Connect and Query

We join four tables to reconstruct the full purchase story for each track:

| Join | Why |
|------|-----|
| `Customer → Invoice` | Links a buyer to their orders |
| `Invoice → InvoiceLine` | Expands each order into individual line items |
| `InvoiceLine → Track` | Resolves the track name for each line item |
| `Track → Album` | Adds the album title |

Results are sorted by customer last name then first name so the output reads like a roster.

In [3]:
conn = sqlite3.connect("Chinook_Sqlite.sqlite")

query = """
SELECT
    Customer.LastName,
    Customer.FirstName,
    Track.Name   AS Name,
    Album.Title  AS Title
FROM Customer
JOIN Invoice      ON Invoice.CustomerId      = Customer.CustomerId
JOIN InvoiceLine  ON InvoiceLine.InvoiceId   = Invoice.InvoiceId
JOIN Track        ON Track.TrackId           = InvoiceLine.TrackId
JOIN Album        ON Album.AlbumId           = Track.AlbumId
ORDER BY
    Customer.LastName  ASC,
    Customer.FirstName ASC
"""

df = pd.read_sql(query, conn)

## Step 4: Display and Clean Up

We display the full DataFrame so every purchase record is visible, then close the database connection. Closing explicitly (rather than relying on garbage collection) ensures any pending read locks are released immediately.

In [4]:
display(df)

conn.close()

,LastName,FirstName,Name,Title
0,Almeida,Roberto,Right Next Door to Hell,Use Your Illusion I
1,Almeida,Roberto,In The Evening,In Through The Out Door
2,Almeida,Roberto,Fool In The Rain,In Through The Out Door
3,Almeida,Roberto,Saudade Dos Aviões Da Panair (Conversando No Bar),Minas
4,Almeida,Roberto,Caso Você Queira Saber,Minas
...,...,...,...,...
2235,Zimmermann,Fynn,Nothin' To Lose,Unplugged [Live]
2236,Zimmermann,Fynn,Since I've Been Loving You,BBC Sessions [Disc 2] [Live]
2237,Zimmermann,Fynn,Going To California,BBC Sessions [Disc 2] [Live]
2238,Zimmermann,Fynn,We're Gonna Groove,Coda
